In [2]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
import pickle

In [3]:
# read in the dataset and create the target variable for classification
df = pd.read_parquet('../datasets/modelling_dataset.parquet')
df['Is_Accident'] = (df['Accident_Count'] > 0).astype(int)

In [4]:
df.head()

,Time_Stamp,Year,Hour,Day_of_Week,Month,Weekend,Holiday,Zone_Int_ID,Amenity,Crossing,...,Precipitation(in),Weather_Clear,Weather_Cloudy,Weather_Dust/Windy,Weather_Rain/Drizzle,Weather_Snow/Ice,Weather_Stormy,Weather_Visibility Issues,Accident_Count,Is_Accident
0,2016-06-14 20:00:00,2016,20,1,6,0,0,0,0.041169,0.233068,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1
1,2016-06-14 20:00:00,2016,20,1,6,0,0,1,0.030181,0.424547,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1,1
2,2016-06-14 20:00:00,2016,20,1,6,0,0,2,0.000000,0.316667,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
3,2016-06-14 20:00:00,2016,20,1,6,0,0,3,0.000000,0.161290,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
4,2016-06-14 20:00:00,2016,20,1,6,0,0,4,0.000000,0.021277,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0


# Approach

In this notebook, I am employing a 2-Step Approach using Logistic Regression as the classifier.

### Step 1: Logistic Regression Classifier

Train a Logistic Regression Classifier on a subsampled training set (1:5 accident to non-accident ratio) 
to predict a binary target: $0$ (No Accident) vs. $1$ (At least one accident). Features are 
standardized using StandardScaler prior to training, as Logistic Regression is sensitive to feature scale.

### Step 2: XGB Regressor (shared with XGBoost pipeline)

Filter dataset to only include rows where Accident_Count > 0. The pre-trained XGBoost Regressor 
predicts expected severity. Target was log-transformed ($\log(1 + \text{count})$) to handle extreme 
outliers (counts up to 105), then converted back with $e^{\text{pred}} - 1$.

### Step 3: Final Risk Score

Model A gives $P(\text{Accident})$. Model B gives $E[\text{Count} \mid \text{Accident}]$. 
Multiply them together to get the final Expected Risk Score:

$$\text{Risk Score} = P(\text{Accident}) \times E[\text{Count} \mid \text{Accident}]$$

The same XGB Regressor is reused here to isolate the effect of the classifier and any difference 
in final Risk Score between this and XGBoost_tuned.ipynb is due to the choice of classification model.

In [5]:
# data split
split_70 = df['Time_Stamp'].quantile(0.70)
split_85 = df['Time_Stamp'].quantile(0.85)

# split into train, validation, and test sets based on time
train_df = df[df['Time_Stamp'] < split_70].copy()
val_df = df[(df['Time_Stamp'] >= split_70) & (df['Time_Stamp'] < split_85)].copy()
test_df = df[df['Time_Stamp'] >= split_85].copy()

In [6]:
from features import engineer_features, regressor_features

# apply to training data first, this computes zone_stats from training only
train_df_eng, zone_stats = engineer_features(train_df)

# pass zone_stats into val and test
val_df_eng,  _ = engineer_features(val_df,  zone_stats=zone_stats)
test_df_eng, _ = engineer_features(test_df, zone_stats=zone_stats)

print("New columns added:", [c for c in train_df_eng.columns if c not in train_df.columns])
print (regressor_features)

New columns added: ['Wind_x_Precip', 'Hour_x_Weekend', 'BadWeather', 'BadWeather_x_Hour', 'Hour_squared', 'WindSpeed_squared', 'Precip_squared', 'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Zone_Mean', 'Zone_Std', 'Zone_Max']
['Hour', 'Day_of_Week', 'Month', 'Weekend', 'Holiday', 'Year', 'Zone_Int_ID', 'Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal', 'City_Houston', 'City_Miami', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Clear', 'Weather_Cloudy', 'Weather_Dust/Windy', 'Weather_Rain/Drizzle', 'Weather_Snow/Ice', 'Weather_Stormy', 'Weather_Visibility Issues', 'Wind_x_Precip', 'Hour_x_Weekend', 'BadWeather', 'BadWeather_x_Hour', 'Hour_squared', 'WindSpeed_squared', 'Precip_squared', 'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Zone_Mean', 'Zone_Std', 'Zone_Max']


In [7]:
# subsampling the training set to provide more balance to the classes for the classification model (not needed for regression)
accidents = train_df_eng[train_df_eng['Is_Accident'] == 1]

non_accidents = train_df_eng[train_df_eng['Is_Accident'] == 0]

non_accidents_sampled = non_accidents.sample(n=len(accidents) * 5, random_state=42)

train_subsampled = pd.concat([accidents, non_accidents_sampled]).sample(frac=1, random_state=42)


In [8]:
# same features as the XGBoost_tuned one
classifier_features = regressor_features

# created X and y for the training, validation, and test sets for classification model
X_train_classifier = train_subsampled[classifier_features]
y_train_classifier = train_subsampled['Is_Accident']

X_val_classifier = val_df_eng[classifier_features]
y_val_classifier = val_df_eng['Is_Accident']

X_test_classifier = test_df_eng[classifier_features]
y_test_classifier = test_df_eng['Is_Accident']

In [9]:
# fit scaler on training data only, apply to val and test
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_classifier)
X_val_scaled = scaler.transform(X_val_classifier)
X_test_scaled = scaler.transform(X_test_classifier)

In [10]:
# train Logistic Regression
# class_weight='balanced' handles remaining imbalance after subsampling
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,          # needs more iterations to converge on large data
    C=0.1,                  # inverse regularization strength
    solver='saga',          # fastest solver for large datasets
    random_state=42,
    n_jobs=-1
)

lr_model.fit(X_train_scaled, y_train_classifier)
print("Logistic Regression trained")

/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression trained


In [22]:
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score

# evaluate on validation set to check performance before final test evaluation
y_val_probs = lr_model.predict_proba(X_val_scaled)[:, 1]
y_val_preds = (y_val_probs >= 0.5).astype(int)  # same threshold as XGBoost

print("Classification Report:")
print(classification_report(y_val_classifier, y_val_preds))

print("Confusion Matrix:")
print(confusion_matrix(y_val_classifier, y_val_preds))

print(f"ROC AUC Score: {round(roc_auc_score(y_val_classifier, y_val_probs), 4)}")

print(f"Average Precision Score: {round(average_precision_score(y_val_classifier, y_val_probs), 4)}")

recall = recall_score(y_val_classifier, y_val_preds)
precision = precision_score(y_val_classifier, y_val_preds)
f1 = f1_score(y_val_classifier, y_val_preds)

print(f"Recall: {round(recall, 4)}")
print(f"Precision: {round(precision, 4)}")
print(f"F1 Score: {round(f1, 4)}")

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.75      0.85    665539
           1       0.16      0.84      0.28     39615

    accuracy                           0.75    705154
   macro avg       0.58      0.79      0.56    705154
weighted avg       0.94      0.75      0.82    705154

Confusion Matrix:
[[496374 169165]
 [  6207  33408]]
ROC AUC Score: 0.8692
Average Precision Score: 0.2675
Recall: 0.8433
Precision: 0.1649
F1 Score: 0.2759


In [12]:
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score
import numpy as np

# use recall as scoring since that's your priority metric
recall_scorer = make_scorer(recall_score)

# combine training and validation sets for hyperparameter tuning with predefined split
X_tuning_classifier = np.vstack([X_train_scaled, X_val_scaled])
y_tuning_classifier = np.concatenate([y_train_classifier.to_numpy(), y_val_classifier.to_numpy()])

# identify training rows (-1) and validation rows (0)
split_index = np.full(X_tuning_classifier.shape[0], -1)
split_index[X_train_scaled.shape[0]:] = 0
pds = PredefinedSplit(test_fold=split_index)

param_grid_r1 = {
    'C': [0.05, 20, 100, 500],
    'penalty': ['l1', 'l2'],
    'class_weight': ['balanced'],
    'solver': ['liblinear'],
    'max_iter': [1000]
}

grid_r1 = GridSearchCV(
    LogisticRegression(random_state=42, n_jobs=-1),
    param_grid_r1,
    cv=pds,
    scoring=recall_scorer,
    verbose=2
)

grid_r1.fit(X_tuning_classifier, y_tuning_classifier)
print(f"Best params: {grid_r1.best_params_}")
print(f"Best recall: {grid_r1.best_score_:.4f}")

Fitting 1 folds for each of 8 candidates, totalling 8 fits


/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


[CV] END C=0.05, class_weight=balanced, max_iter=1000, penalty=l1, solver=liblinear; total time= 3.3min


/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


[CV] END C=0.05, class_weight=balanced, max_iter=1000, penalty=l2, solver=liblinear; total time=  15.7s


...

[CV] END C=500, class_weight=balanced, max_iter=1000, penalty=l1, solver=liblinear; total time= 3.6min


/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


[CV] END C=500, class_weight=balanced, max_iter=1000, penalty=l2, solver=liblinear; total time=  14.9s


/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Best params: {'C': 20, 'class_weight': 'balanced', 'max_iter': 1000, 'penalty': 'l2', 'solver': 'liblinear'}
Best recall: 0.8441


In [18]:
# training regression with best hyperparameters from grid search
lr_model_tuned = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,          # needs more iterations to converge on large data
    C=20,
    penalty='l2',                  # inverse regularization strength
    solver='liblinear',          # fastest solver for large datasets
    random_state=42,
    n_jobs=-1
)

lr_model_tuned.fit(X_train_scaled, y_train_classifier)
print("Tuned Logistic Regression trained")

/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/MSE446/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Tuned Logistic Regression trained


In [23]:
# Evaluate tuned model on validation set

from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score

# evaluate on validation set to check performance before final test evaluation
y_val_probs = lr_model_tuned.predict_proba(X_val_scaled)[:, 1]
y_val_preds = (y_val_probs >= 0.5).astype(int)  # same threshold as XGBoost

print("Classification Report:")
print(classification_report(y_val_classifier, y_val_preds))

print("Confusion Matrix:")
print(confusion_matrix(y_val_classifier, y_val_preds))

print(f"ROC AUC Score: {round(roc_auc_score(y_val_classifier, y_val_probs), 4)}")

print(f"Average Precision Score: {round(average_precision_score(y_val_classifier, y_val_probs), 4)}")

recall = recall_score(y_val_classifier, y_val_preds)
precision = precision_score(y_val_classifier, y_val_preds)
f1 = f1_score(y_val_classifier, y_val_preds)

print(f"Recall: {round(recall, 4)}")
print(f"Precision: {round(precision, 4)}")
print(f"F1 Score: {round(f1, 4)}")

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.75      0.85    665539
           1       0.17      0.84      0.28     39615

    accuracy                           0.75    705154
   macro avg       0.58      0.79      0.56    705154
weighted avg       0.94      0.75      0.82    705154

Confusion Matrix:
[[496336 169203]
 [  6174  33441]]
ROC AUC Score: 0.8693
Average Precision Score: 0.2678
Recall: 0.8441
Precision: 0.165
F1 Score: 0.2761


In [24]:
# Evaluate tuned model on test set
y_test_probs = lr_model_tuned.predict_proba(X_test_scaled)[:, 1]
y_test_preds = (y_test_probs >= 0.4).astype(int)

print("Classification Report:")
print(classification_report(y_test_classifier, y_test_preds))

print("Confusion Matrix:")
print(confusion_matrix(y_test_classifier, y_test_preds))

print(f"ROC AUC Score: {round(roc_auc_score(y_test_classifier, y_test_probs), 4)}")

recall = recall_score(y_test_classifier, y_test_preds)
precision = precision_score(y_test_classifier, y_test_preds)
f1 = f1_score(y_test_classifier, y_test_preds)

print(f"Recall: {round(recall, 4)}")
print(f"Precision: {round(precision, 4)}")
print(f"F1 Score: {round(f1, 4)}")

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.68      0.80    670990
           1       0.12      0.87      0.21     34322

    accuracy                           0.69    705312
   macro avg       0.56      0.77      0.51    705312
weighted avg       0.95      0.69      0.77    705312

Confusion Matrix:
[[453891 217099]
 [  4507  29815]]
ROC AUC Score: 0.8461
Recall: 0.8687
Precision: 0.1208
F1 Score: 0.212
